# 🚀 ComfyUI + MiniMax-H3 on Google Colab (A100 GPU Edition)

Google AI Pro 等のプランで付与される **Colab Compute Units (CU)** を活用して、強力な最新動画生成モデル **MiniMax-H3 (Hailuo)** を A100 (40GB VRAM) 環境で動かすためのオールインワン検証テンプレートです。

### 💡 特徴・ポイント
- **A100 GPU 最適化**: MiniMax-H3 の大規模モデル (`int8_convrot` 等) + Text Encoder + Audio/Video VAE をストレスなくロード。
- **Google Drive 完全永続化 (モデルキャッシュ & 出力動画)**:
  - モデル（約20GB）を `MyDrive/ComfyUI_Models/` にキャッシュし、2回目以降のDLをスキップ（起動待ち0秒）。
  - 生成された動画・画像は **`MyDrive/ComfyUI_Outputs/` に自動保存**。ランタイムが切断・終了しても成果物が消えません。
- **ComfyUI 公式最新サポート**: v0.3.0 以降のネイティブ MiniMax-H3 ノード対応。
- **Cloudflare Tunnel (無料・トークン不要)**: ngrok 不要ですぐにセキュアな一時公開 URL (`trycloudflare.com`) を自動発行。

> ⚠️ **注意**: ノートブック上部のメニュー「ランタイム」→「ランタイムのタイプを変更」から、**GPU (A100)** が選択されていることを確認してください。

## Step 1: 環境確認 & Google Drive マウント (永続化連携)

In [ ]:
# GPU の確認 (A100 がアサインされているか確認)
!nvidia-smi

# モデルキャッシュ & 出力動画を Google Drive に永続保存
USE_GOOGLE_DRIVE = True  # @param {type:"boolean"}

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully!")

## Step 2: ComfyUI 本体 & 必須カスタムノードのセットアップ

最新の ComfyUI および ComfyUI-Manager をセットアップします。

In [ ]:
import os

# 作業ディレクトリへ移動
%cd /content

# ComfyUI 本体のクローン (最新版)
if not os.path.exists("/content/ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI.git
else:
    %cd /content/ComfyUI
    !git pull
    %cd /content

# 依存ライブラリのインストール
%cd /content/ComfyUI
!pip install -q -r requirements.txt
!pip install -q sageattention huggingface_hub

# ComfyUI-Manager のインストール
%cd /content/ComfyUI/custom_nodes
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-Manager"):
    !git clone https://github.com/ltdrdata/ComfyUI-Manager.git

# MiniMax-H3 Easy / 補助ノード (任意で便利に利用可能)
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-MiniMaxH3-Easy"):
    !git clone https://github.com/kijai/ComfyUI-MiniMaxH3-Easy.git || true

%cd /content/ComfyUI

## Step 3: MiniMax-H3 モデル & 出力先の設定 (Google Drive 連携)

- `USE_GOOGLE_DRIVE = True` の場合：
  - **モデルキャッシュ**: `/content/drive/MyDrive/ComfyUI_Models/` (次回以降 DL スキップ)
  - **動画出力先**: `/content/drive/MyDrive/ComfyUI_Outputs/` (インスタンス終了後も成果物を保持)

In [ ]:
import os
from huggingface_hub import hf_hub_download

# モデルおよび出力先のディレクトリ準備
if USE_GOOGLE_DRIVE:
    MODELS_BASE = "/content/drive/MyDrive/ComfyUI_Models"
    OUTPUT_DIR = "/content/drive/MyDrive/ComfyUI_Outputs"
    os.makedirs(MODELS_BASE, exist_ok=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # 1. 出力フォルダを Google Drive にシンボリックリンク
    local_output = "/content/ComfyUI/output"
    if os.path.exists(local_output) and not os.path.islink(local_output):
        !rm -rf {local_output}
    if not os.path.exists(local_output):
        os.symlink(OUTPUT_DIR, local_output)

    # 2. モデルフォルダを Google Drive にシンボリックリンク
    for sub in ["diffusion_models", "text_encoders", "vae"]:
        drive_sub = os.path.join(MODELS_BASE, sub)
        local_sub = os.path.join("/content/ComfyUI/models", sub)
        os.makedirs(drive_sub, exist_ok=True)
        if os.path.exists(local_sub) and not os.path.islink(local_sub):
            !rm -rf {local_sub}
        if not os.path.exists(local_sub):
            os.symlink(drive_sub, local_sub)
    print(f"📁 Google Drive キャッシュ & 出力先を連携しました:\n  - Models: {MODELS_BASE}\n  - Outputs: {OUTPUT_DIR}")
else:
    MODELS_BASE = "/content/ComfyUI/models"

REPO_ID = "Comfy-Org/MiniMax-H3"

files_to_download = [
    ("diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors", "diffusion_models"),
    ("text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "text_encoders"),
    ("vae/minimax_h3_video_vae_fp16.safetensors", "vae"),
    ("vae/minimax_h3_audio_vae_fp32.safetensors", "vae")
]

print("📥 MiniMax-H3 モデルファイルの確認 & ダウンロードを開始...")
for filename, folder in files_to_download:
    target_path = os.path.join(MODELS_BASE, filename)
    if os.path.exists(target_path) and os.path.getsize(target_path) > 1000000:
        print(f"⚡ キャッシュ検出 (DLスキップ): {filename}")
    else:
        print(f"⬇️ ダウンロード中: {filename} ...")
        hf_hub_download(
            repo_id=REPO_ID,
            filename=filename,
            local_dir=MODELS_BASE,
            local_dir_use_symlinks=False
        )

print("✅ すべての MiniMax-H3 モデルの準備が完了しました！")

## Step 4: ComfyUI 起動 & Cloudflare Tunnel 経由でアクセス

バックグラウンドで ComfyUI を起動し、Cloudflare Tunnel (`trycloudflare.com`) の安全なパブリック URL を発行して**起動状態を維持**します。
表示された `https://xxxx.trycloudflare.com` のリンクをクリックすると ComfyUI の WebUI が開きます。

> 💡 **「すべて実行」でも安心**: 本セルはトンネルが稼働している間ループ待機するため、勝手に Step 5 のシャットダウンへ進むことはありません。

In [ ]:
import subprocess
import threading
import time
import re

# Cloudflared のダウンロード
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

# ComfyUI をバックグラウンドで起動
print("⚡ Starting ComfyUI...")
%cd /content/ComfyUI
comfy_proc = subprocess.Popen([
    "python", "main.py",
    "--listen", "127.0.0.1",
    "--port", "8188",
    "--highvram"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# ComfyUI の起動ログを別スレッドで監視
def log_comfy():
    for line in iter(comfy_proc.stdout.readline, ''):
        print("[ComfyUI]", line, end="")

threading.Thread(target=log_comfy, daemon=True).start()

# 少し待機して Cloudflared トンネルを起動
time.sleep(5)
print("🌐 Starting Cloudflare Tunnel...")
tunnel_proc = subprocess.Popen([
    "/content/cloudflared", "tunnel",
    "--url", "http://127.0.0.1:8188"
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

url_found = False
for line in iter(tunnel_proc.stdout.readline, ''):
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match and not url_found:
        url = match.group(0)
        print("\n" + "="*60)
        print(f"🎉 ComfyUI is LIVE: {url}")
        print("="*60 + "\n")
        print("💡 サーバーを稼働維持しています。(終了したい場合は本セルの停止ボタンを押すか Step 5 を実行)")
        url_found = True

# トンネルプロセスが生きている間はセルを終了させず維持
try:
    tunnel_proc.wait()
except KeyboardInterrupt:
    print("\n⏹️ サーバーを停止しました。")
    comfy_proc.terminate()
    tunnel_proc.terminate()

## Step 5: (作業終了時) ランタイムの切断 & CU 消費停止

動画生成の検証が完了したら、以下のセルの **`CONFIRM_DISCONNECT = True`** にチェックを入れて実行します。
生成された動画ファイルは既に Google Drive（`ComfyUI_Outputs/`）に保存されているため、安全にインスタンスを破棄して **Compute Units (CU) のアイドル消費を即座にストップ** できます。

In [ ]:
# 安全装置: 誤実行で即座に切断されないよう確認フラグを設けています
CONFIRM_DISCONNECT = False  # @param {type:"boolean"}

if CONFIRM_DISCONNECT:
    from google.colab import runtime
    print("🛑 Disconnecting runtime to save Compute Units...")
    runtime.unassign()
else:
    print("ℹ️ 切断は実行されませんでした。インスタンスを終了する場合は CONFIRM_DISCONNECT を True にして実行してください。")